In [1]:
from odps_client import get_odps_sql_result_as_df
from datetime import datetime, timedelta

daily_high_search_volume_theshold = 470 / 14
daily_low_search_volume_theshold = 2

last_n_days = 60
ds_yesterday = (datetime.now() - timedelta(days=1)).strftime("%Y%m%d")
last_n_days_ago = (datetime.now() - timedelta(days=last_n_days)).strftime("%Y%m%d")

top_query = f"""
SELECT  query
        ,COUNT(DISTINCT CASE    WHEN envent_type = 'impression' THEN cust_id END) AS searched_users
        ,COUNT(DISTINCT CASE    WHEN envent_type = 'impression' THEN CONCAT(time,cust_id) END) AS search_cnt
        ,COUNT(DISTINCT CASE    WHEN envent_type = 'click' THEN CONCAT(time,cust_id) END) AS click_cnt
        ,ROUND(AVG(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END),1) AS avg_click_index
        ,ROUND(MAX(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END),1) AS max_click_index
        ,ROUND(MIN(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END),1) AS min_click_index
        ,ROUND(STDDEV(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END),1) AS std_click_index
        ,PERCENTILE(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END,0.5) AS p50_click_index
        ,PERCENTILE(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END,0.75) AS p75_click_index
        ,PERCENTILE(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END,0.9) AS p90_click_index
        ,CASE
            WHEN COUNT(DISTINCT CASE WHEN envent_type = 'impression' THEN CONCAT(time,cust_id) END) > {last_n_days*daily_high_search_volume_theshold} THEN '高频搜索词'
            WHEN COUNT(DISTINCT CASE WHEN envent_type = 'impression' THEN CONCAT(time,cust_id) END) <= {last_n_days*daily_low_search_volume_theshold} THEN '低频搜索词'
            ELSE '中频搜索词'
         END AS 搜索频次标签
        ,COUNT(DISTINCT CASE WHEN envent_type = 'click' THEN CONCAT(time,cust_id) END) * 1.0 / COUNT(DISTINCT CASE WHEN envent_type = 'impression' THEN CONCAT(time,cust_id) END) as ctr_uv
        ,COUNT(distinct ds) as 有搜索天数
        ,CASE
            WHEN COUNT(DISTINCT CASE WHEN envent_type = 'click' THEN CONCAT(time,cust_id) END) * 1.0 / COUNT(DISTINCT CASE WHEN envent_type = 'impression' THEN CONCAT(time,cust_id) END) > 0.25 THEN '高点击率词'
            ELSE '低点击率词'
         END AS 点击率标签
FROM    summerfarm_tech.app_log_search_detail_di
WHERE   ds BETWEEN '{last_n_days_ago}' and '{ds_yesterday}'
GROUP BY query
ORDER BY searched_users DESC;
"""

top_query_df = get_odps_sql_result_as_df(sql=top_query)
top_query_df.head(20)

2025-01-16 22:49:44 - INFO - Thread count: 20
2025-01-16 22:50:06 - INFO - Tunnel session created: <InstanceDownloadSession id=20250116225005ad19481a00590f36 project_name=summerfarm_ds instance_id=20250116144944871gef80qp2im1>
2025-01-16 22:50:07 - INFO - sql:

SELECT  query
        ,COUNT(DISTINCT CASE    WHEN envent_type = 'impression' THEN cust_id END) AS searched_users
        ,COUNT(DISTINCT CASE    WHEN envent_type = 'impression' THEN CONCAT(time,cust_id) END) AS search_cnt
        ,COUNT(DISTINCT CASE    WHEN envent_type = 'click' THEN CONCAT(time,cust_id) END) AS click_cnt
        ,ROUND(AVG(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END),1) AS avg_click_index
        ,ROUND(MAX(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END),1) AS max_click_index
        ,ROUND(MIN(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END),1) AS min_click_index
        ,ROUND(STDDEV(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END),1) AS s

,query,searched_users,search_cnt,click_cnt,avg_click_index,max_click_index,min_click_index,std_click_index,p50_click_index,p75_click_index,p90_click_index,搜索频次标签,ctr_uv,有搜索天数,点击率标签
0,草莓,16780,349471,106159,7.3,172.0,0.0,8.5,5.0,10.0,17.0,高频搜索词,0.303771,60,高点击率词
1,芒果,12880,266012,68911,7.4,214.0,0.0,7.9,5.0,10.0,16.0,高频搜索词,0.259052,60,高点击率词
2,牛奶,11483,136237,39440,14.2,328.0,0.0,14.1,10.0,20.0,29.0,高频搜索词,0.289496,60,高点击率词
3,安佳,10194,77815,27993,5.6,144.0,0.0,7.5,3.0,6.0,15.0,高频搜索词,0.359738,60,高点击率词
4,黄油,8908,103510,18008,18.3,213.0,0.0,17.9,12.0,28.0,43.0,高频搜索词,0.173974,60,低点击率词
5,奶油,8544,106529,21370,21.8,255.0,0.0,25.0,13.0,31.0,54.0,高频搜索词,0.200603,60,低点击率词
6,蓝莓,7363,61929,25150,2.5,93.0,0.0,4.3,2.0,3.0,5.0,高频搜索词,0.406110,60,高点击率词
7,安佳淡奶油,6793,35473,18277,1.4,159.0,0.0,5.2,1.0,1.0,2.0,高频搜索词,0.515237,60,高点击率词
8,柠檬,6699,64956,22798,8.4,76.0,0.0,6.8,7.0,13.0,18.0,高频搜索词,0.350976,60,高点击率词
9,铁塔,5676,27153,13856,1.5,83.0,0.0,5.3,0.0,1.0,3.0,高频搜索词,0.510294,60,高点击率词


In [2]:
from sls_client import get_sls_data_by_query
from datetime import datetime, timedelta
import pandas as pd

# 设置pandas显示选项以展示更多内容
pd.set_option("display.max_rows", 100)  # 显示最多100行
pd.set_option("display.max_columns", None)  # 显示所有列
pd.set_option("display.width", 1000)  # 设置显示宽度
pd.set_option("display.max_colwidth", 100)  # 设置列最大宽度

import sqlite3


def get_user_variant_of_date_from_sls(
    day: datetime, check_if_local_exist: bool = True
) -> pd.DataFrame:
    db_file_name = f"./data/search_ab_user_variant.db"
    table_name = f"search_ab_user_variant_{day.strftime('%Y%m%d')}"
    conn = sqlite3.connect(db_file_name)
    cursor = conn.cursor()

    if check_if_local_exist:
        try:
            query = f"SELECT * FROM {table_name}"
            df = pd.read_sql_query(query, conn)
            conn.close()
            return df
        except pd.io.sql.DatabaseError:
            pass

    query = f"""
type:a and ap:/product/ and pageName:/search/goods|
select regexp_replace(ap, '\d+','{{digit}}') as api,pageName as page_ame,
json_extract_scalar(json_extract_scalar(ai, '$.qh.xm-ab-exp'),'$[1].experimentId') experiment_id,
type,uid,date_format(__time__, '%Y%m%d') as ds,count(1) search_times,
array_join(array_sort(array_agg(distinct json_extract_scalar(json_extract_scalar(ai, '$.qh.xm-ab-exp'),'$[1].variantId'))),',') variant_list
from log group by 1,2,3,4,5,6 limit 1000000"""
    from_time = day.replace(hour=0, minute=0, second=0, microsecond=0)
    to_time = day.replace(hour=23, minute=59, second=59, microsecond=999999)
    _df = get_sls_data_by_query(
        query=query,
        project="xianmu-front-end-log",
        logstore="xm-mall",
        from_time=from_time,
        to_time=to_time,
    )

    _df["search_times"] = _df["search_times"].fillna(1).astype(int)

    _df["variant_list"] = _df["variant_list"].fillna("none")

    if not _df.empty:
        _df.drop(columns=["__source__", "__time__"], inplace=True)
        _df.to_sql(table_name, conn, if_exists="replace", index=False)
    conn.close()
    return _df


all_user_variant_df = pd.DataFrame()
start_date = datetime(2025, 1, 15)
end_date = datetime.now()
current_date = start_date
while current_date <= end_date:
    is_today = current_date.strftime("%Y%m%d") == end_date.strftime("%Y%m%d")
    if is_today:
        print(f"今天的数据还未完整，跳过:{current_date}")
        break
    df = get_user_variant_of_date_from_sls(current_date, check_if_local_exist=True)
    all_user_variant_df = pd.concat([all_user_variant_df, df], ignore_index=True)
    current_date += timedelta(days=1)

all_user_variant_df.head(10)

今天的数据还未完整，跳过:2025-01-16 00:00:00


,api,page_ame,experiment_id,type,uid,ds,search_times,variant_list
0,/product/{digit}/{digit},/search/goods,product_search_rerank,a,341054,20250115,1,V1
1,/product/{digit}/{digit},/search/goods,product_search_rerank,a,401321,20250115,1,V3
2,/product/{digit}/{digit},/search/goods,product_search_rerank,a,255992,20250115,6,V3
3,/product/{digit}/{digit},/search/goods,product_search_rerank,a,38421,20250115,14,V1
4,/product/{digit}/{digit},/search/goods,product_search_rerank,a,350290,20250115,11,V1
5,/product/{digit}/{digit},/search/goods,product_search_rerank,a,431399,20250115,8,V2
6,/product/{digit}/{digit},/search/goods,product_search_rerank,a,461448,20250115,3,V3
7,/product/{digit}/{digit},/search/goods,product_search_rerank,a,182650,20250115,8,V3
8,/product/{digit}/{digit},/search/goods,product_search_rerank,a,279059,20250115,7,V3
9,/product/{digit}/{digit},/search/goods,product_search_rerank,a,153695,20250115,2,V3


In [3]:
# idx:4,name:徐州奶油草莓 净重3-3.2斤/一级/单果10g+/板装,pid:goods,sku:5442468008,salePrice:69.5,pdid:702,stock:10000,ext:cross;idx:5,name:徐州奶油草莓 280G*1盒/一级/4*6/ ,pid:goods,sku:5442468073,salePrice:16.5,pdid:702,stock:10000,ext:cross;idx:6,name:徐州奶油草莓 净重2.8-3斤/一级/单果10g+/ 10盒,pid:goods,sku:5442468518,salePrice:74.5,pdid:702,stock:10000,ext:cross

view_query = """
type:view and pageName:/search/goods |
select   regexp_extract(sku_item, 'idx:(\d+)', 1) AS idx,
  regexp_extract(sku_item, 'name:([^,]+)', 1) AS name,
  regexp_extract(sku_item, 'sku:([\dA-Z]+)', 1) AS sku,
  regexp_extract(sku_item, 'pid:([^,]+)', 1) AS pid,
  regexp_extract(sku_item, 'pdid:(\d+)', 1) AS pdid,search_query,type,uid,ds
from(
select uid,date_format(__time__, '%Y%m%d') ds,bid_list.sku_item,
url_extract_parameter(split_part(url,'#/',2), 'pdName') AS search_query,type
from log,unnest(split(bid,';')) as bid_list(sku_item) limit 10000000)
having pid = 'goods'
"""


def get_user_sku_view_of_date_from_sls(
    day: datetime, check_if_local_exist: bool = True
) -> pd.DataFrame:
    db_file_name = f"./data/search_ab_user_sku_view.db"
    table_name = f"user_sku_view_{day.strftime('%Y%m%d')}"
    conn = sqlite3.connect(db_file_name)
    cursor = conn.cursor()

    if check_if_local_exist:
        try:
            query = f"SELECT * FROM {table_name}"
            df = pd.read_sql_query(query, conn)
            conn.close()
            return df
        except pd.io.sql.DatabaseError:
            pass

    from_time = day.replace(hour=0, minute=0, second=0, microsecond=0)
    to_time = day.replace(hour=23, minute=59, second=59, microsecond=999999)
    _df = get_sls_data_by_query(
        query=view_query,
        project="xianmu-front-end-log",
        logstore="xm-mall",
        from_time=from_time,
        to_time=to_time,
    )

    if not _df.empty:
        _df.drop(columns=["__source__", "__time__"], inplace=True)
        _df.to_sql(table_name, conn, if_exists="replace", index=False)
    conn.close()
    return _df


all_user_sku_view_df = pd.DataFrame()
current_date = start_date
while current_date <= end_date:
    is_today = current_date.strftime("%Y%m%d") == end_date.strftime("%Y%m%d")
    if is_today:
        print(f"今天的数据还未完整，跳过:{current_date}")
        break
    df = get_user_sku_view_of_date_from_sls(current_date, check_if_local_exist=True)
    all_user_sku_view_df = pd.concat([all_user_sku_view_df, df], ignore_index=True)
    current_date += timedelta(days=1)

即将获取数据: =====> 2025-01-15 00:00:00 2025-01-15 23:59:59.999999 xm-mall: 
type:view and pageName:/search/goods |
select   regexp_extract(sku_item, 'idx:(\d+)', 1) AS idx,
  
>=====数条数:229377
今天的数据还未完整，跳过:2025-01-16 00:00:00


,idx,name,sku,pid,pdid,search_query,type,uid,ds


In [4]:
from sls_client import get_sls_raw_data_by_query

click_query = """
type:cl and pageName:/search/goods |
select   coalesce(idx,regexp_extract(sku_item, 'idx:(\d+)', 1)) AS idx,
  coalesce(name,regexp_extract(sku_item, 'name:([^,]+)', 1)) AS name,
  coalesce(sku,regexp_extract(sku_item, 'sku:([\dA-Z]+)', 1)) AS sku,
  coalesce(pid,regexp_extract(sku_item, 'pid:([^,]+)', 1)) AS pid,
  coalesce(pdid,regexp_extract(sku_item, 'pdid:(\d+)', 1)) AS pdid,bid,
ds,search_query,type,uid,page_name,sku_item,coalesce(linkInfo,url)linkInfo from(
select uid,date_format(__time__, '%Y%m%d') ds,replace(replace(split_part(url_decode(split_part(url,'#/',2)),'?',2),'=',':'),'&',',') url,
bid_list.sku_item,pageName as page_name,bid,idx,name,sku,pid,pdid,linkInfo,
url_extract_parameter(split_part(url,'#/',2), 'pdName') AS search_query,rqCount rq_count,type
from log,unnest(split(bid,';')) as bid_list(sku_item) limit 1000000)"""

# click_query = "type:cl and pageName:/search/goods"


def get_user_sku_click_of_date_from_sls(
    day: datetime, check_if_local_exist: bool = True
) -> pd.DataFrame:
    db_file_name = f"./data/search_ab_user_sku_click.db"
    table_name = f"user_sku_click_{day.strftime('%Y%m%d')}"
    conn = sqlite3.connect(db_file_name)
    cursor = conn.cursor()

    if check_if_local_exist:
        try:
            query = f"SELECT * FROM {table_name}"
            df = pd.read_sql_query(query, conn)
            conn.close()
            return df
        except pd.io.sql.DatabaseError:
            pass

    from_time = day.replace(hour=0, minute=0, second=0, microsecond=0)
    to_time = day.replace(hour=23, minute=59, second=59, microsecond=999999)
    _df = get_sls_raw_data_by_query(
        query=click_query,
        project="xianmu-front-end-log",
        logstore="xm-mall",
        from_time=from_time,
        to_time=to_time,
    )

    if not _df.empty:
        _df.drop(
            columns=[
                "__source__",
                "__time__",
                "userAgent",
                "url",
                "__topic__",
                "__tag__:__client_ip__",
                "__tag__:__receive_time__",
                "__time_ns_part__",
            ],
            inplace=True,
            errors="ignore",
        )
        _df["ds"] = day.strftime("%Y%m%d")
        _df.to_sql(table_name, conn, if_exists="replace", index=False)
    conn.close()
    return _df


all_user_sku_click_df = pd.DataFrame()
current_date = start_date
while current_date <= end_date:
    is_today = current_date.strftime("%Y%m%d") == end_date.strftime("%Y%m%d")
    if is_today:
        print(f"今天的数据还未完整，跳过:{current_date}")
        break
    df = get_user_sku_click_of_date_from_sls(current_date, check_if_local_exist=True)
    all_user_sku_click_df = pd.concat([all_user_sku_click_df, df], ignore_index=True)
    current_date += timedelta(days=1)

all_user_sku_click_df.head(10)

即将获取数据: =====>from_time:2025-01-15 00:00:00, to_time:2025-01-15 23:59:59.999999, logstore:xm-mall, query:
type:cl and pageName:/search/goods |
select   coalesce(idx,regexp_extract(sku_item, 'idx:(\d+)', 1)) AS idx,
  coalesce(name,regexp_extract(sku_item,
>=====数据条数:28101
今天的数据还未完整，跳过:2025-01-16 00:00:00


,idx,name,sku,pid,pdid,bid,ds,search_query,type,uid,page_name,sku_item,linkInfo
0,5,海南水仙芒 净重12-12.5斤/三级/单果约250g+(内置问题百分之15以内不售后),5440754143,唤起购买,819,undefined,20250115,凯特,cl,372545,/search/goods,undefined,"name:searchGoods,pdName:凯特,type:2"
1,null,阶梯价步进器,605352552140,加购弹窗,null,undefined,20250115,丝诺,cl,503691,/search/goods,undefined,"name:searchGoods,pdName:丝诺,type:2"
2,null,阶梯价步进器,605352552140,加购弹窗,null,undefined,20250115,丝诺,cl,503691,/search/goods,undefined,"name:searchGoods,pdName:丝诺,type:2"
3,null,阶梯价步进器,605352552140,加购弹窗,null,undefined,20250115,丝诺,cl,503691,/search/goods,undefined,"name:searchGoods,pdName:丝诺,type:2"
4,null,阶梯价步进器,605352552140,加购弹窗,null,undefined,20250115,丝诺,cl,503691,/search/goods,undefined,"name:searchGoods,pdName:丝诺,type:2"
5,null,丝诺含乳脂植脂奶油 907G*12盒,605352552140,加购弹窗,12262,"name:丝诺含乳脂植脂奶油 907G*12盒,pid:加购弹窗,sku:605352552140,pdid:12262,stock:4",20250115,丝诺,cl,503691,/search/goods,"name:丝诺含乳脂植脂奶油 907G*12盒,pid:加购弹窗,sku:605352552140,pdid:12262,stock:4","name:searchGoods,pdName:丝诺,type:2"
6,null,加入购物车,605352552140,加购弹窗,12262,"name:加入购物车,pid:加购弹窗,sku:605352552140,pdid:12262,stock:4",20250115,丝诺,cl,503691,/search/goods,"name:加入购物车,pid:加购弹窗,sku:605352552140,pdid:12262,stock:4","name:searchGoods,pdName:丝诺,type:2"
7,null,加入购物车,605352552140,加购弹窗,12262,"name:加入购物车,pid:加购弹窗,sku:605352552140,pdid:12262,stock:4",20250115,丝诺,cl,503691,/search/goods,"name:加入购物车,pid:加购弹窗,sku:605352552140,pdid:12262,stock:4","name:searchGoods,pdName:丝诺,type:2"
8,4,安佳再制切达奶酪_橙色 84片*1包,1256524478,唤起购买,719,undefined,20250115,芝士片,cl,85440,/search/goods,undefined,"name:searchGoods,pdName:芝士片,type:2"
9,11,爱护咖啡用浓缩植脂奶油 1L*12盒,15487486557,goods,1634,"idx:11,name:爱护咖啡用浓缩植脂奶油 1L*12盒,pid:goods,sku:15487486557,salePrice:359,pdid:1634,stock:10000,ext...",20250115,维益咖啡奶油,cl,108331,/search/goods,"idx:11,name:爱护咖啡用浓缩植脂奶油 1L*12盒,pid:goods,sku:15487486557,salePrice:359,pdid:1634,stock:10000,ext...","name:searchGoods,pdName:维益咖啡奶油,type:2"


In [5]:
import re

all_user_sku_click_explored = []
pattern = re.compile(r'idx:(?P<idx>\d+).*?name:(?P<name>[^,]+).*?pid:(?P<pid>[^,]+).*?sku:(?P<sku>[^,]+).*?pdid:(?P<pdid>[^,]+)')

for index, row in all_user_sku_click_df.iterrows():
    _dict = row.to_dict()
    bid = _dict["bid"]
    for bid_item in bid.split(";"):
        sku_info = {}
        sku_info.update(_dict)
        search_query=_dict["linkInfo"]
        sku_info["search_query"] = re.search(r'pdName:([^,]+)', search_query).group(1)
        sku_info["bid"] = bid_item
        if 'undefined' in bid_item:
            all_user_sku_click_explored.append(sku_info)
        else:
            try:
                idx = pdid = sku = pid = name = None
                
                idx_match = re.search(r'idx:(\d+)', bid_item)
                if idx_match:
                    idx = idx_match.group(1)
                    
                pdid_match = re.search(r'pdid:(\d+)', bid_item)
                if pdid_match:
                    pdid = pdid_match.group(1)
                    
                sku_match = re.search(r'sku:([\dA-Za-z]+)', bid_item)
                if sku_match:
                    sku = sku_match.group(1)
                    
                pid_match = re.search(r'pid:([^,]+)', bid_item)
                if pid_match:
                    pid = pid_match.group(1)
                    
                name_match = re.search(r'name:([^,]+)', bid_item)
                if name_match:
                    name = name_match.group(1)
                    
                sku_info.update({
                    "idx": idx,
                    "pdid": pdid, 
                    "sku": sku,
                    "pid": pid,
                    "name": name
                })
                all_user_sku_click_explored.append(sku_info)
            except Exception as e:
                print(e, bid_item)
                raise e

all_user_sku_click_explored_df = pd.DataFrame(all_user_sku_click_explored)
all_user_sku_click_explored_df[['bid','sku','name','idx','pid','pdid','linkInfo','search_query']].head(5)

,bid,sku,name,idx,pid,pdid,linkInfo,search_query
0,undefined,5440754143,海南水仙芒 净重12-12.5斤/三级/单果约250g+(内置问题百分之15以内不售后),5,唤起购买,819,"name:searchGoods,pdName:凯特,type:2",凯特
1,undefined,605352552140,阶梯价步进器,null,加购弹窗,null,"name:searchGoods,pdName:丝诺,type:2",丝诺
2,undefined,605352552140,阶梯价步进器,null,加购弹窗,null,"name:searchGoods,pdName:丝诺,type:2",丝诺
3,undefined,605352552140,阶梯价步进器,null,加购弹窗,null,"name:searchGoods,pdName:丝诺,type:2",丝诺
4,undefined,605352552140,阶梯价步进器,null,加购弹窗,null,"name:searchGoods,pdName:丝诺,type:2",丝诺


In [6]:
print(all_user_variant_df.columns)
print(all_user_sku_view_df.columns)
print(all_user_sku_click_df.columns)

all_sku_view_data_df = all_user_sku_view_df[
    [
        "idx",
        "name",
        "sku",
        "pid",
        "pdid",
        "uid",
        "ds",
        "search_query",
        "type",
    ]
].merge(
    all_user_variant_df[["uid", "ds", "variant_list", "search_times"]],
    on=["uid", "ds"],
    how="left",
)


Index(['api', 'page_ame', 'experiment_id', 'type', 'uid', 'ds', 'search_times', 'variant_list'], dtype='object')
Index(['idx', 'name', 'sku', 'pid', 'pdid', 'search_query', 'type', 'uid', 'ds'], dtype='object')
Index(['idx', 'name', 'sku', 'pid', 'pdid', 'bid', 'ds', 'search_query', 'type', 'uid', 'page_name', 'sku_item', 'linkInfo'], dtype='object')


In [7]:
all_user_sku_click_explored_df.groupby("pid").size().reset_index(name="count").sort_values(
    "count", ascending=False
).head(10)

,pid,count
1,加购弹窗,11067
3,唤起购买,9186
0,goods,7845
2,唤起提醒,3


In [8]:
user_click_with_variant_df = all_user_sku_click_explored_df.merge(
    all_user_variant_df[["uid", "ds", "variant_list"]],
    on=["uid", "ds"],
    how="left",
)

user_click_with_variant_df["action_type"] = user_click_with_variant_df.apply(
    lambda row: (
        "加入购物车"
        if row["pid"] == "加购弹窗" and row["name"] == "加入购物车"
        else "商品详情" if row["pid"] == "goods" else row["pid"]
    ),
    axis=1,
)
user_click_with_variant_df['idx'] = user_click_with_variant_df['idx'].fillna(-1)
user_click_with_variant_df['idx'] = user_click_with_variant_df['idx'].replace('null', -1).astype(int)

In [9]:
import pandasql

user_click_with_variant_statistics_df = pandasql.sqldf(
"""
select uid,variant_list,ds,coalesce(b.搜索频次标签,'其他') as 搜索频次标签,
count(case when action_type='商品详情' then 1 end) as 商品详情cnt,
count(case when action_type='加入购物车' then 1 end) as 加入购物车cnt,
count(case when action_type='唤起购买' and variant_list is not null then 1 end) as 唤起购买cnt,
count(case when (action_type='唤起购买' and variant_list is not null) or action_type='商品详情' then 1 end) as 总点击cnt,
count(case when (action_type='唤起购买' and variant_list is not null or action_type='商品详情') and idx>=0 and idx<=5 then 1 end) as 首屏总点击cnt,
round(avg(case when action_type='商品详情' or action_type='唤起购买' then idx end),1) as avg点击位置,
coalesce(max(case when action_type='商品详情' or action_type='唤起购买' then idx end),100) as max点击位置,
coalesce(min(case when action_type='商品详情' or action_type='唤起购买' then idx end),100) as min点击位置,
count(distinct sku) 点击SKU_cnt,
count(distinct search_query) 搜索词cnt                        
from user_click_with_variant_df a
left join top_query_df b on a.search_query = b.query
group by uid,variant_list,ds,coalesce(b.搜索频次标签,'其他')
""")

user_click_with_variant_statistics_df.head(5)

,uid,variant_list,ds,搜索频次标签,商品详情cnt,加入购物车cnt,唤起购买cnt,总点击cnt,首屏总点击cnt,avg点击位置,max点击位置,min点击位置,点击SKU_cnt,搜索词cnt
0,100041,V3,20250115,中频搜索词,5,0,0,5,5,0.0,0,0,4,4
1,100041,V3,20250115,高频搜索词,10,0,0,10,8,6.0,41,0,8,6
2,100322,V3,20250115,中频搜索词,2,0,0,2,1,10.0,20,0,2,2
3,100745,V3,20250115,中频搜索词,0,0,1,1,1,0.0,0,0,1,1
4,100745,V3,20250115,高频搜索词,0,0,2,2,2,2.0,2,2,1,1


In [10]:
null_search_query_df = pandasql.sqldf(
    """select case when search_query is null or search_query = 'null' then 'null-search-query' else 'normal' end has_search_query,
                                    count(1) cnt from all_sku_view_data_df group by 1"""
)
null_search_query_df
# 约有2.3%的数据没有搜索词，这部分需要过滤掉
all_sku_view_data_df = all_sku_view_data_df[
    all_sku_view_data_df["search_query"] != "null"
]

In [11]:
all_sku_view_data_df["idx"] = all_sku_view_data_df["idx"].fillna(-1).astype(int)
user_view_with_variant_statistics_df = pandasql.sqldf(
"""
select uid,ds,variant_list,coalesce(b.搜索频次标签,'其他') as 搜索频次标签,min(a.search_query) sample_query,
count(1) as 商品查看cnt,
count(distinct sku) as 查看SKU_cnt,
count(distinct search_query) as 查看搜索词cnt,
max(idx) as max查看位置,
max(search_times) as 搜索翻页数cnt
from all_sku_view_data_df a
left join top_query_df b on a.search_query = b.query
group by uid,variant_list,ds,coalesce(b.搜索频次标签,'其他')
"""
)


In [12]:
all_data_df = user_view_with_variant_statistics_df.merge(
    user_click_with_variant_statistics_df, on=["uid", "ds", "variant_list","搜索频次标签"], how="left"
)

all_data_df["商品详情cnt"] = all_data_df["商品详情cnt"].fillna(0).astype(int)
all_data_df["加入购物车cnt"] = all_data_df["加入购物车cnt"].fillna(0).astype(int)
all_data_df["唤起购买cnt"] = all_data_df["唤起购买cnt"].fillna(0).astype(int)
all_data_df["首屏总点击cnt"] = all_data_df["首屏总点击cnt"].fillna(0).astype(int)
# all_data_df["min点击位置"] = all_data_df["min点击位置"].fillna(100).astype(int)
all_data_df["总点击cnt"] = all_data_df["总点击cnt"].fillna(0).astype(int)
all_data_df["用户是否点击"] = all_data_df["总点击cnt"].apply(lambda x: 1 if x > 0 else 0)
# all_data_df["avg点击位置"] = all_data_df["avg点击位置"].fillna(0.0).astype(float)
all_data_df["sku_click_rate"] = (all_data_df['商品详情cnt']*1.00/all_data_df['商品查看cnt']).fillna(0).round(5).astype(float)
all_data_df["add_cart_rate"] = (all_data_df['加入购物车cnt']/all_data_df['商品查看cnt']).fillna(0).round(5).astype(float)
all_data_df["popup_click_rate"] = (all_data_df['唤起购买cnt']/all_data_df['商品查看cnt']).fillna(0).round(5).astype(float)

In [13]:
from IPython.core.display import HTML
import pandas as pd

css = """
<link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/bootstrap@4.0.0/dist/css/bootstrap.min.css" integrity="sha384-Gn5384xqQ1aoWXA+058RXPxPg6fy4IWvTNh0E263XmFcJlSAwiGgFAW/dAiS6JXm" crossorigin="anonymous">
<style type=\"text/css\">
#abTesting table,#abTesting .table {
    color: #333;
    font-family: unset;
    font-size: 12px;
    line-height: 1.5;
    width: 95vw;
    border-collapse:
    collapse; 
    border-spacing: 0;
    font-family: "SF Pro SC", "SF Pro Text", "SF Pro Icons", "PingFang SC", "Helvetica Neue", "Helvetica", "Arial", sans-serif;
}

body{
    padding-left: 1rem;
    padding-top: 1vh;
}

tr{
    border-bottom: 1px solid #C1C3D1;
}

tr:nth-child(even) {
    background-color: #F8F8F8;
}

#abTesting td, #abTesting th {
    /* border: 1px solid transparent; No more visible border */
    height: 30px;
    padding: 0.2rem;
}

#abTesting table tbody td,#abTesting .table tbody td{
    padding: 0.1rem .75rem;
    vertical-align: middle;
}

th {
    background-color: #DFDFDF; /* Darken header a bit */
    font-weight: bolder;
    font-size: larger;
    color: #000;
    text-align: center;
}
</style>
"""


def display_p_value_below_005(row: pd.Series, p_value_col_name: str = "p_value"):
    p_value = row[p_value_col_name]
    color = "black"
    if p_value is not None and p_value <= 0.05:
        color = "red"
    return f"""<span style='font-weight:bolder;color:{color};'>{p_value}</span>"""


def display_diff_to_v2(row: pd.Series, metric: str = "diff_to_v2%"):
    diff = row[metric]
    color = "green"
    if diff is not None and float(diff) > 0.0:
        color = "red"
    return f"""<span style='font-weight:bolder;color:{color};'>{diff:.4f} %</span>"""


def dataframe_to_html(df: pd.DataFrame, title: str):
    df_to_display = df.copy()

    df_to_display["p_value"] = df_to_display.apply(display_p_value_below_005, axis=1)
    df_to_display["diff_to_v2%"] = df_to_display.apply(display_diff_to_v2, axis=1)

    html_df = df_to_display.to_html(
        escape=False, index=False, classes="table dataframe"
    )
    html_content = f"""<html><head><meta charset="UTF-8">
    <meta name="title" content="{title}">
    {css}
    </head><body>
    <h2>{title}</h2>
    <h4>当P-value <= 0.05时表示实验结果统计学显著</h4>
    <span>统计学显著时，既可能表示该试验组是好于对照组，也可能是坏于对照组</span>
    <div id="abTesting">{html_df}</div></body></html>"""

    return html_content

In [14]:
import pandas as pd
from scipy.stats import ttest_ind


def calculate_p_values(
    df: pd.DataFrame,
    metric: str = "商品详情cnt",
    control_variant: str = "V2",
) -> pd.DataFrame:
    """
    Calculate p-values for each combination of category1 and page_name.
    Compares metric between control group (V1) and each of V2, V3, V4.

    Parameters:
    - df (pd.DataFrame): The input DataFrame containing A/B test data.
    - metric (str): The metric column to be analyzed (default is 'added_quantity').

    Returns:
    - pd.DataFrame: A DataFrame with category1, page_name, variant, p-value, and statistics columns.
    """
    p_values = []

    control = df[df["variant_list"] == control_variant][metric]
    control_avg = control.mean()

    for variant in ["V1", "V2", "V3", "V4", "V5"]:
        test_group = df[df["variant_list"] == variant]
        test = test_group[metric]

        print(
            f'variant:{variant}, test_group ds length: {len(test_group["ds"].unique())}'
        )
        if len(test_group["ds"].unique()) <= 0:
            continue

        # Calculate statistics
        stats = {
            "均值": round(test.mean(), 4),
            "std": round(test.std(), 4),
            f"diff_to_{control_variant}%".lower(): round(
                100.00 * (test.mean() - control_avg) / control_avg, 2
            ),
            "q50": test.quantile(0.5),
            "q75": test.quantile(0.75),
            "q90": test.quantile(0.9),
            "q95": test.quantile(0.95),
            "q97": test.quantile(0.97),
            "q99": test.quantile(0.99),
            "q995": test.quantile(0.995),
            "max": test.max(),
            "日均总数": round(test.sum() / len(test_group["ds"].unique())),
            "日均实验UV": round(
                len(test_group[["uid", "ds"]].drop_duplicates())
                / len(test_group["ds"].unique())
            ),
            "日均转化UV": round(
                len(test_group[test_group[metric] > 0][["uid", "ds"]].drop_duplicates())
                / len(test_group["ds"].unique())
            ),
            "日期范围": f"{test_group['ds'].min()}~{test_group['ds'].max()}".replace(
                "2025", ""
            ),
            "metric": metric,
        }

        # Ensure both groups have enough data for a valid t-test
        if len(control) > 1 and len(test) > 1:
            # Perform independent t-test
            stat, p_val = ttest_ind(control, test, equal_var=False, nan_policy="omit")
            p_values.append(
                {
                    "variant_list": variant,
                    "p_value": round(p_val, 4),
                    **stats,
                }
            )
            print(f"stat:{stat}")
        else:
            # Not enough data for statistical testing
            p_values.append(
                {
                    "variant_list": variant,
                    "p_value": None,
                    **stats,
                }
            )

    return pd.DataFrame(p_values)

In [15]:
# Define the desired order for sorting
variant_order = ["V1", "V2", "V3", "V4", "V5"]


# Create a custom sort key function
def sort_key(variant):
    # Split variant by commas
    parts = variant.split(",")
    # Determine the order based on the first variant in the list
    if parts[0] in variant_order:
        return variant_order.index(parts[0])
    else:
        return len(variant_order)  # Place all other variants after V1, V2, V3, V4


metrics_list = [
    "sku_click_rate",
    "avg点击位置",
    "min点击位置",
    "总点击cnt",
    "首屏总点击cnt",
    "用户是否点击",
    "商品查看cnt",
]
all_p_values_df = pd.DataFrame()
for metric in metrics_list:
    all_p_values_of_same_metric_df = pd.DataFrame()
    for label, group_df in all_data_df.groupby("搜索频次标签"):
        if label == "其他":
            print("ignore 其他")
            continue
        p_values_df = calculate_p_values(
            group_df,
            metric=metric,
        )

        p_values_df["搜索频次"] = label

        p_values_df["variant_list"] = p_values_df["variant_list"].apply(
            lambda x: x if x in variant_order else "X_" + x
        )
        p_values_df = p_values_df.sort_values(
            by="variant_list", key=lambda x: x.map(sort_key)
        )
        p_values_df["variant_list"] = p_values_df["variant_list"].str.replace("X_", "")
        all_p_values_of_same_metric_df = pd.concat(
            [all_p_values_of_same_metric_df, p_values_df], ignore_index=True
        )
        all_p_values_df = pd.concat([all_p_values_df, p_values_df], ignore_index=True)

    title = f"搜索AB--{metric}_p-value分布-{p_values_df.iloc[0]['日期范围']}"

    html_content = dataframe_to_html(df=all_p_values_of_same_metric_df, title=title)
    file_path = f"./data/{title}.html"

    # 保存HTML到本地文件：
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(html_content)

    print(f"写入HTML成功！{file_path}")


title_all = f"搜索AB--指标全集_p-value分布-{all_p_values_df.iloc[0]['日期范围']}"
html_content = dataframe_to_html(df=all_p_values_df, title=title_all)
file_path = f"./data/{title_all}.html"

# 保存HTML到本地文件：
with open(file_path, "w", encoding="utf-8") as f:
    f.write(html_content)

print(f"写入HTML成功！{file_path}")
all_p_values_df

variant:V1, test_group ds length: 1
stat:0.5655168316824272
variant:V2, test_group ds length: 1
stat:0.0
variant:V3, test_group ds length: 1
stat:0.0330144560692853
variant:V4, test_group ds length: 0
variant:V5, test_group ds length: 1
variant:V1, test_group ds length: 1
stat:0.22451696859153783
variant:V2, test_group ds length: 1
stat:0.0
variant:V3, test_group ds length: 1
stat:-1.1040243114848172
variant:V4, test_group ds length: 0
variant:V5, test_group ds length: 0
variant:V1, test_group ds length: 1
stat:-0.42156138202267246
variant:V2, test_group ds length: 1
stat:0.0
variant:V3, test_group ds length: 1
stat:-0.030219005032667855
variant:V4, test_group ds length: 0
variant:V5, test_group ds length: 0
写入HTML成功！./data/搜索AB--sku_click_rate_p-value分布-0115~0115.html
variant:V1, test_group ds length: 1
stat:-0.6433857220460656
variant:V2, test_group ds length: 1
stat:0.0
variant:V3, test_group ds length: 1
stat:1.579650840855374
variant:V4, test_group ds length: 0
variant:V5, test_gr

,variant_list,p_value,均值,std,diff_to_v2%,q50,q75,q90,q95,q97,q99,q995,max,日均总数,日均实验UV,日均转化UV,日期范围,metric,搜索频次
0,V1,0.5719,0.0458,0.0852,-6.63,0.00,0.066670,0.142860,0.25000,0.250,0.333330,0.500000,0.66667,21,463,179,0115~0115,sku_click_rate,中频搜索词
1,V2,1.0000,0.0491,0.0943,0.00,0.00,0.062500,0.200000,0.25000,0.250,0.368269,0.460713,1.00000,25,511,192,0115~0115,sku_click_rate,中频搜索词
2,V3,0.9737,0.0489,0.0873,-0.32,0.00,0.067235,0.188096,0.25000,0.250,0.333330,0.400000,0.87500,72,1479,565,0115~0115,sku_click_rate,中频搜索词
3,V5,NaN,0.0000,NaN,-100.00,0.00,0.000000,0.000000,0.00000,0.000,0.000000,0.000000,0.00000,0,1,0,0115~0115,sku_click_rate,中频搜索词
4,V1,0.8225,0.0401,0.0807,-4.47,0.00,0.043480,0.146014,0.25000,0.250,0.296186,0.351664,0.50000,7,179,59,0115~0115,sku_click_rate,低频搜索词
5,V2,1.0000,0.0420,0.0793,0.00,0.00,0.051315,0.158978,0.25000,0.250,0.250000,0.267500,0.50000,8,187,64,0115~0115,sku_click_rate,低频搜索词
6,V3,0.2703,0.0497,0.0946,18.24,0.00,0.071430,0.177275,0.25000,0.250,0.391333,0.500000,1.00000,31,614,228,0115~0115,sku_click_rate,低频搜索词
7,V1,0.6734,0.0420,0.0718,3.20,0.00,0.055955,0.131880,0.21773,0.250,0.260229,0.333330,0.66667,44,1056,475,0115~0115,sku_click_rate,高频搜索词
8,V2,1.0000,0.0407,0.0715,0.00,0.00,0.052630,0.138506,0.20000,0.250,0.294934,0.333330,0.50000,45,1095,476,0115~0115,sku_click_rate,高频搜索词
9,V3,0.9759,0.0408,0.0714,0.18,0.00,0.055560,0.125000,0.20000,0.250,0.250000,0.333330,0.75000,135,3313,1433,0115~0115,sku_click_rate,高频搜索词


In [16]:
from odps_client import get_odps_sql_result_as_df, write_pandas_df_into_odps

partition_spec = f"pt={datetime.now().strftime('%Y%m%d')}"

write_pandas_df_into_odps(
    df=all_user_variant_df,
    table_name="summerfarm_ds.temp_search_ab_all_data_df",
    partition_spec=partition_spec,
    overwrite=True,
    lifecycle=30,
)

# 只分析哪些进入过搜索页面的用户的订单转化结果

start_date_str = start_date.strftime("%Y-%m-%d")

order_query = f"""
with user_orders as (
    SELECT  m_id
        ,total_price
        ,order_no
        ,DATE_FORMAT(order_time,'yyyyMMdd') as order_date
    FROM    summerfarm_tech.ods_orders_df
    WHERE   ds = MAX_PT('summerfarm_tech.ods_orders_df')
    AND     status IN (2,3,6)
    AND     order_time >= '{start_date_str} 00:00:00'
    AND     m_size = '单店'
),user_variants as (
    select ds as event_date,uid,variant_list
    from summerfarm_ds.temp_search_ab_all_data_df
    where pt=max_pt('summerfarm_ds.temp_search_ab_all_data_df')
)
select a.event_date,a.uid,a.variant_list,sum(b.total_price) as order_gmv,
    count(b.order_no) as order_cnt,round(sum(b.total_price)/count(b.order_no),2) as avg_order_gmv
from user_variants a
left join user_orders b on a.uid=b.m_id and a.event_date=b.order_date
group by a.event_date,a.uid,a.variant_list
"""

user_orders_df = get_odps_sql_result_as_df(order_query)
user_orders_df.head(2)

2025-01-16 22:50:22 - INFO - DaraFrame字段合集:uid,type,page_ame,variant_list,pt,experiment_id,search_times,create_time,ds,api
2025-01-16 22:50:25 - INFO - Tunnel session created: <TableUploadSession id=20250116225025135ac90b0062599e project=summerfarm_ds table=temp_search_ab_all_data_df partition_spec=pt=20250116>
2025-01-16 22:50:28 - INFO - 成功写入odps:summerfarm_ds.temp_search_ab_all_data_df, partition_spec:pt=20250116, attemp:0
2025-01-16 22:50:36 - INFO - Tunnel session created: <InstanceDownloadSession id=202501162250364af8c20b006240b7 project_name=summerfarm_ds instance_id=20250116145028726gwo15pkujm16>
2025-01-16 22:50:37 - INFO - sql:

with user_orders as (
    SELECT  m_id
        ,total_price
        ,order_no
        ,DATE_FORMAT(order_time,'yyyyMMdd') as order_date
    FROM    summerfarm_tech.ods_orders_df
    WHERE   ds = MAX_PT('summerfarm_tech.ods_orders_df')
    AND     status IN (2,3,6)
    AND     order_time >= '2025-01-15 00:00:00'
    AND     m_size = '单店'
),user_variant

,event_date,uid,variant_list,order_gmv,order_cnt,avg_order_gmv
0,20250115,418287,V3,245,1,245
1,20250115,193887,V3,180,1,180


In [17]:
user_orders_df["order_gmv"]=user_orders_df["order_gmv"].astype(float)
user_orders_df["avg_order_gmv"]=user_orders_df["avg_order_gmv"].astype(float)
user_orders_df["order_cnt"]=user_orders_df["order_cnt"].astype(int)
user_orders_df[user_orders_df["order_gmv"]>0]["order_gmv"].describe()

count     3482.000000
mean       687.862275
std       1432.450452
min          6.000000
25%        184.925000
50%        348.000000
75%        725.750000
max      46800.000000
Name: order_gmv, dtype: float64

In [18]:
print(
    f"所有订单的分布:\n",
    user_orders_df[user_orders_df["order_gmv"]>0]["order_gmv"].quantile(
        [0.5, 0.75, 0.95, 0.99, 0.995, 0.996, 0.997, 0.999, 1]
    ),
)


# 这里排除哪些高单价的订单，否则对于数据分析来说不好处理。
user_orders_below_6k_df = user_orders_df[user_orders_df["order_gmv"] <= 6000]
print(
    "排除高单价的订单后的分布:\n",
    user_orders_below_6k_df["order_gmv"].quantile(
        [0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
    ),
)

所有订单的分布:
 0.500      348.000
0.750      725.750
0.950     2124.650
0.990     5501.700
0.995     8095.775
0.996     9461.400
0.997    10681.300
0.999    18283.300
1.000    46800.000
Name: order_gmv, dtype: float64
排除高单价的订单后的分布:
 0.01      31.793
0.05      85.970
0.25     184.000
0.50     345.000
0.75     709.000
0.95    1940.000
0.99    3778.400
Name: order_gmv, dtype: float64


In [19]:
user_orders_below_6k_df["order_gmv"] = user_orders_below_6k_df["order_gmv"].fillna(0)
user_orders_below_6k_df["order_gmv"] = user_orders_below_6k_df["order_gmv"].astype(float)

user_orders_below_6k_df["avg_order_gmv"] = user_orders_below_6k_df["avg_order_gmv"].fillna(0.0)
user_orders_below_6k_df["avg_order_gmv"] = user_orders_below_6k_df["avg_order_gmv"].astype(float)

user_orders_below_6k_df["category1"] = "ignore"
user_orders_below_6k_df["page_name"] = "ignore"

user_orders_during_ab_df = user_orders_below_6k_df[
    user_orders_below_6k_df["variant_list"].isin(["V1", "V2", "V3", "V4", "V5"])
]
user_orders_during_ab_df.rename(columns={"event_date": "ds"}, inplace=True)

all_order_pvalue_df = pd.DataFrame()
for metric in ["order_gmv", "avg_order_gmv", "order_cnt"]:
    gmv_df = calculate_p_values(df=user_orders_during_ab_df, metric=metric)
    display(gmv_df)
    all_order_pvalue_df = pd.concat([all_order_pvalue_df, gmv_df], ignore_index=True)

title = f"搜索AB--订单转化p-value分布-{all_order_pvalue_df.iloc[0]['日期范围']}"
html_content = dataframe_to_html(df=all_order_pvalue_df, title=title)
file_path = f"./data/{title}.html"

# 保存HTML到本地文件：
with open(file_path, "w", encoding="utf-8") as f:
    f.write(html_content)

print(f"写入HTML成功！{file_path}")
display(all_order_pvalue_df)

variant:V1, test_group ds length: 1
stat:0.0505838181744665
variant:V2, test_group ds length: 1
stat:0.0
variant:V3, test_group ds length: 1
stat:-1.6565526897449314
variant:V4, test_group ds length: 0
variant:V5, test_group ds length: 0


/var/folders/cl/v_4j9fbj5nn_jj6q9d3r5ggc0000gn/T/ipykernel_34540/403049697.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  user_orders_below_6k_df["order_gmv"] = user_orders_below_6k_df["order_gmv"].fillna(0)
/var/folders/cl/v_4j9fbj5nn_jj6q9d3r5ggc0000gn/T/ipykernel_34540/403049697.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  user_orders_below_6k_df["order_gmv"] = user_orders_below_6k_df["order_gmv"].astype(float)
/var/folders/cl/v_4j9fbj5nn_jj6q9d3r5ggc0000gn/T/ipykernel_34540/403049697.py:4: S

,variant_list,p_value,均值,std,diff_to_v2%,q50,q75,q90,q95,q97,q99,q995,max,日均总数,日均实验UV,日均转化UV,日期范围,metric
0,V1,0.9597,567.0392,674.5472,-0.32,338.0,668.625,1292.1,1983.85,2370.250,3505.27,3823.5400,5700.0,393525,694,694,0115~0115,order_gmv
1,V2,1.0000,568.8713,684.5096,0.00,324.5,644.250,1210.8,1900.00,2318.020,3388.05,4216.5500,5700.0,406174,714,714,0115~0115,order_gmv
2,V3,0.0978,619.6877,761.4528,8.93,354.5,740.000,1337.9,1927.15,2573.755,4295.50,5275.2925,5825.0,1261684,2036,2036,0115~0115,order_gmv


variant:V1, test_group ds length: 1
stat:0.3006328956827612
variant:V2, test_group ds length: 1
stat:0.0
variant:V3, test_group ds length: 1
stat:-1.2003545960051314
variant:V4, test_group ds length: 0
variant:V5, test_group ds length: 0


,variant_list,p_value,均值,std,diff_to_v2%,q50,q75,q90,q95,q97,q99,q995,max,日均总数,日均实验UV,日均转化UV,日期范围,metric
0,V1,0.7637,465.9717,537.9676,-1.86,287.835,549.750,963.30,1470.750,1930.40,2670.56,3247.025,5700.0,323384,694,694,0115~0115,avg_order_gmv
1,V2,1.0000,474.8014,564.0537,0.00,282.750,572.625,1034.95,1459.550,1881.10,2781.13,3249.570,5700.0,339008,714,714,0115~0115,avg_order_gmv
2,V3,0.2302,504.8216,605.1687,6.32,310.950,600.000,1042.75,1534.375,1917.25,2994.75,4256.175,5700.0,1027817,2036,2036,0115~0115,avg_order_gmv


variant:V1, test_group ds length: 1
stat:0.17761550921702146
variant:V2, test_group ds length: 1
stat:0.0
variant:V3, test_group ds length: 1
stat:-0.8272605506891597
variant:V4, test_group ds length: 0
variant:V5, test_group ds length: 0


,variant_list,p_value,均值,std,diff_to_v2%,q50,q75,q90,q95,q97,q99,q995,max,日均总数,日均实验UV,日均转化UV,日期范围,metric
0,V1,0.8591,1.2190,0.5408,-0.41,1.0,1.0,2.0,2.0,3.0,3.0,4.0,5,846,694,694,0115~0115,order_cnt
1,V2,1.0000,1.2241,0.5298,0.00,1.0,1.0,2.0,2.0,3.0,3.0,4.0,5,874,714,714,0115~0115,order_cnt
2,V3,0.4082,1.2436,0.5777,1.60,1.0,1.0,2.0,2.0,3.0,3.0,4.0,7,2532,2036,2036,0115~0115,order_cnt


写入HTML成功！./data/搜索AB--订单转化p-value分布-0115~0115.html


,variant_list,p_value,均值,std,diff_to_v2%,q50,q75,q90,q95,q97,q99,q995,max,日均总数,日均实验UV,日均转化UV,日期范围,metric
0,V1,0.9597,567.0392,674.5472,-0.32,338.000,668.625,1292.10,1983.850,2370.250,3505.27,3823.5400,5700.0,393525,694,694,0115~0115,order_gmv
1,V2,1.0000,568.8713,684.5096,0.00,324.500,644.250,1210.80,1900.000,2318.020,3388.05,4216.5500,5700.0,406174,714,714,0115~0115,order_gmv
2,V3,0.0978,619.6877,761.4528,8.93,354.500,740.000,1337.90,1927.150,2573.755,4295.50,5275.2925,5825.0,1261684,2036,2036,0115~0115,order_gmv
3,V1,0.7637,465.9717,537.9676,-1.86,287.835,549.750,963.30,1470.750,1930.400,2670.56,3247.0250,5700.0,323384,694,694,0115~0115,avg_order_gmv
4,V2,1.0000,474.8014,564.0537,0.00,282.750,572.625,1034.95,1459.550,1881.100,2781.13,3249.5700,5700.0,339008,714,714,0115~0115,avg_order_gmv
5,V3,0.2302,504.8216,605.1687,6.32,310.950,600.000,1042.75,1534.375,1917.250,2994.75,4256.1750,5700.0,1027817,2036,2036,0115~0115,avg_order_gmv
6,V1,0.8591,1.2190,0.5408,-0.41,1.000,1.000,2.00,2.000,3.000,3.00,4.0000,5.0,846,694,694,0115~0115,order_cnt
7,V2,1.0000,1.2241,0.5298,0.00,1.000,1.000,2.00,2.000,3.000,3.00,4.0000,5.0,874,714,714,0115~0115,order_cnt
8,V3,0.4082,1.2436,0.5777,1.60,1.000,1.000,2.00,2.000,3.000,3.00,4.0000,7.0,2532,2036,2036,0115~0115,order_cnt


In [20]:
all_p_values_df.to_csv(
    f"./data/搜索AB--所有指标p-value分布-{all_p_values_df.iloc[0]['日期范围']}.csv",
    index=False,
)
all_order_pvalue_df.to_csv(
    f"./data/搜索AB--订单转化p-value分布-{all_p_values_df.iloc[0]['日期范围']}.csv",
    index=False,
)